# Вероятности места игрока в турнире РТТ

Отдельный режим для турниров, которые ещё не начались или находятся в процессе проведения. Он использует готовый prediction bundle только для чтения и хранит страницы RTT в отдельном `tournament_analysis_cache`.

- до публикации состава берутся заявки, прогнозируются участники ОТ, посев и сетка;
- после публикации используются официальный состав и, когда она доступна, официальная сетка;
- сеяные разводятся по секциям согласно Регламенту РТТ, учитываются `X`;
- результат содержит распределение мест, очки и матожидание очков;
- мега-режим сравнивает несколько текущих турниров по матожиданию очков.

In [4]:
from __future__ import annotations

import base64
from datetime import date
import html
from io import BytesIO
import json
import os
from pathlib import Path
import subprocess
import sys

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'scripts' / 'prediction_runtime.py').exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import prediction_runtime as pr
from rtt_predictor.tournament_data import load_snapshot
from rtt_predictor.tournament_mode import (
    analyze_tournament,
    cached_registered_tour_ids,
    current_rating_age_groups,
    eligible_tour_ids_from_master,
    mega_summary,
    prepare_registration_scenario,
)

CACHE_DIR = ROOT / 'tournament_analysis_cache'
MASTER_PATH = ROOT / 'data' / 'tournaments_master.xlsx'
bundle = pr.load_prediction_bundle()
print(f"Bundle: {bundle.get('model_name', 'production_model')} | {bundle.get('bundle_path')}")
print(f"Последний матч в данных: {pd.Timestamp(bundle['long_feat']['match_date'].max()).date()}")

Bundle: production_catboost | c:\Users\Anton\Desktop\PythonProjects\Предиктор матчей\assembled_predictor\prediction_bundle.joblib
Последний матч в данных: 2026-08-16


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.8.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
try:
    _PREVIOUS_RUN_BUTTON._click_handlers.callbacks.clear()
except (NameError, AttributeError):
    pass

LAST_ANALYSIS = None
LAST_ANALYSES = []
_RUNNING = False

mode_box = widgets.ToggleButtons(
    options=['Один турнир', 'Мега: сравнить турниры'],
    value='Один турнир',
    description='Режим:',
)
tour_ids_box = widgets.Text(
    description='Турниры:',
    placeholder='Например: 305996 или 305996, 306048',
    layout=widgets.Layout(width='650px'),
)
player_box = widgets.Combobox(
    options=pr.player_options(bundle),
    description='Игрок:',
    ensure_option=False,
    layout=widgets.Layout(width='650px'),
)
iterations_box = widgets.Dropdown(
    options=[2_000, 5_000, 10_000, 20_000, 50_000],
    value=20_000,
    description='Симуляций:',
)
refresh_box = widgets.Checkbox(value=True, description='Обновить отдельный кэш RTT')
auto_mega_box = widgets.Checkbox(
    value=False,
    description='Мега: все доступные турниры + фактические заявки',
    indent=False,
)
run_button = widgets.Button(description='Рассчитать', button_style='primary', icon='play')
progress = widgets.IntProgress(value=0, min=0, max=100, description='Прогресс:')
status_html = widgets.HTML(value='<b>Готово к запуску.</b>')
result_html = widgets.HTML(value='', layout=widgets.Layout(overflow_x='auto'))
opponent_box = widgets.Dropdown(options=[], description='H2H:', disabled=True, layout=widgets.Layout(width='520px'))
h2h_button = widgets.Button(description='Показать стандартный H2H', button_style='info', disabled=True)
h2h_result_html = widgets.HTML(value='', layout=widgets.Layout(overflow_x='auto'))

def _tour_ids():
    values = [value.strip() for value in tour_ids_box.value.replace(';', ',').split(',') if value.strip()]
    if mode_box.value.startswith('Мега') and auto_mega_box.value and not values:
        inferred_ages = current_rating_age_groups(bundle, player_box.value.strip()) if player_box.value.strip() else []
        registered = cached_registered_tour_ids(CACHE_DIR, player_box.value.strip()) if player_box.value.strip() else []
        discovered = eligible_tour_ids_from_master(MASTER_PATH, age_group=inferred_ages)
        values = list(dict.fromkeys([*registered, *discovered]))
    return list(dict.fromkeys(values))

def _refresh(ids):
    if not ids:
        return ''
    stable_python = Path(os.environ.get('LOCALAPPDATA', '')) / 'Programs' / 'Python' / 'Python312' / 'python.exe'
    fetch_python = str(stable_python) if stable_python.exists() else sys.executable
    command = [fetch_python, '-S', str(ROOT / 'scripts' / 'fetch_rtt_tournament.py')]
    for tour_id in ids:
        command.extend(['--tour-id', tour_id])
    command.extend(['--cache-dir', str(CACHE_DIR), '--concurrency', '4', '--timeout-seconds', '35'])
    env = os.environ.copy()
    venv_site = ROOT / '.venv' / 'Lib' / 'site-packages'
    if venv_site.exists():
        env['PYTHONPATH'] = str(venv_site) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    browser_cache = ROOT / 'tmp' / 'ms-playwright'
    if browser_cache.exists():
        env.setdefault('PLAYWRIGHT_BROWSERS_PATH', str(browser_cache))
    completed = subprocess.run(command, cwd=ROOT, env=env, text=True, capture_output=True, timeout=max(180, 55 * len(ids)))
    if completed.returncode != 0:
        try:
            payload = json.loads(completed.stdout)
            errors = [str(row.get('error', '')) for row in payload.get('results', []) if not row.get('ok')]
            detail = ' | '.join(dict.fromkeys(error for error in errors if error))
        except Exception:
            detail = (completed.stdout + '\n' + completed.stderr).strip()[-1200:]
        status_html.value = '<b style="color:#b26a00">RTT API не ответил; пробую отдельный кэш.</b>'
        return detail
    return ''

def _progress(done, total, message):
    progress.max = max(total, 1)
    progress.value = min(done, progress.max)
    progress.description = 'H2H:'
    status_html.value = f'<b>{html.escape(message)}</b>'

def _table_html(frame):
    return frame.to_html(index=False, border=0, classes='rtt-table', justify='left')

def _analysis_html(analysis):
    global LAST_ANALYSIS
    LAST_ANALYSIS = analysis
    snapshot = analysis.snapshot
    target_id = analysis.target_player.player_id
    entry_scenario = ('Виртуальная заявка' if 'virtual_registration' in snapshot.player_source else 'Фактическая заявка')
    expected = analysis.simulation.expected_points.get(target_id)
    expected_html = (f'<b>матожидание очков:</b> {expected:.2f}' if expected is not None else '<b>Матожидание очков:</b> нет таблицы для этой категории')
    parts = [
        f'<h3>{html.escape(snapshot.title or str(snapshot.tour_id))}</h3>'
        f'<p><b>Турнир:</b> {snapshot.tour_id} &nbsp; <b>Статус:</b> {html.escape(snapshot.status)} &nbsp; <b>Кэш:</b> {html.escape(snapshot.fetched_at)} &nbsp; '
        f'<b>Категория:</b> {html.escape(snapshot.category)} &nbsp; <b>Возраст:</b> {html.escape(snapshot.age_group)}</p>'
        f'<p><b>Сценарий:</b> {entry_scenario}; <b>состав:</b> {len(analysis.players)} игроков, {html.escape(snapshot.player_source)}; '
        f'<b>сетка:</b> {"официальная" if analysis.simulation.draw_is_fixed else "смоделированная по правилам РТТ"}; ' + expected_html + '</p>'
    ]
    distribution = analysis.distribution_table().copy()
    distribution = distribution.rename(columns={
        'place': 'Место', 'probability_pct': 'Вероятность, %', 'points': 'Очки',
        'expected_points_contribution': 'Вклад в матожидание',
    })
    parts.append(_table_html(distribution[['Место', 'Вероятность, %', 'Очки', 'Вклад в матожидание']].round(3)))
    plot_data = analysis.distribution_table()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(plot_data['place'], plot_data['probability_pct'], color='#315efb')
    ax.set_ylabel('Вероятность, %')
    ax.set_xlabel('Итоговое место')
    ax.set_title(f'Распределение мест: {analysis.target_player.name}')
    ax.grid(axis='y', alpha=.2)
    image_buffer = BytesIO()
    fig.tight_layout()
    fig.savefig(image_buffer, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    image_data = base64.b64encode(image_buffer.getvalue()).decode('ascii')
    parts.append(f'<img alt="Распределение мест" src="data:image/png;base64,{image_data}" style="max-width:100%;height:auto">')
    opponents = analysis.opponents_table()
    if not opponents.empty:
        parts.append('<h4>Возможные соперники — переход к H2H ниже</h4>')
        opponent_table = opponents[['opponent', 'encounter_probability_pct', 'target_h2h_win_probability_pct']].rename(columns={
            'opponent': 'Соперник', 'encounter_probability_pct': 'Вероятность встречи, %',
            'target_h2h_win_probability_pct': 'Вероятность победы в H2H, %',
        }).round(2)
        parts.append(_table_html(opponent_table))
        opponent_box.options = [(row.opponent, row.opponent_id) for row in opponents.itertuples()]
        opponent_box.disabled = False
        h2h_button.disabled = False
    if analysis.warnings:
        parts.append('<h4>Предупреждения</h4><ul>' + ''.join(f'<li>{html.escape(item)}</li>' for item in analysis.warnings) + '</ul>')
    return ''.join(parts)

def _run(_):
    global LAST_ANALYSES, _RUNNING
    if _RUNNING:
        return
    _RUNNING = True
    run_button.disabled = True
    result_html.value = ''
    try:
        ids = _tour_ids()
        player_name = player_box.value.strip()
        if not ids:
            raise ValueError('Укажите номер турнира; в мега-режиме можно включить автоподбор из master.')
        if not player_name:
            raise ValueError('Укажите игрока.')
        progress.value = 0
        refresh_detail = ''
        if refresh_box.value:
            status_html.value = '<b>Обновляю отдельный кэш RTT…</b>'
            refresh_detail = _refresh(ids)
        snapshots = []
        snapshot_errors = []
        for tour_id in ids:
            try:
                snapshots.append(load_snapshot(CACHE_DIR, tour_id))
            except Exception as exc:
                snapshot_errors.append(f'Турнир {tour_id}: {exc}')
        if not snapshots:
            detail = (f' Причина обновления: {refresh_detail}' if refresh_detail else '')
            raise RuntimeError('RTT API сейчас недоступен, а для выбранных турниров ещё нет отдельного кэша.' + detail)
        result_parts = ['<style>.rtt-table{border-collapse:collapse;margin:8px 0 18px}.rtt-table th,.rtt-table td{padding:5px 9px;border-bottom:1px solid #ddd;text-align:left}</style>']
        if snapshot_errors:
            result_parts.append('<p><b>Не загружены:</b> ' + html.escape(' | '.join(snapshot_errors)) + '</p>')
        if mode_box.value == 'Один турнир':
            if len(snapshots) != 1:
                raise ValueError('В режиме одного турнира укажите ровно один номер.')
            scenario = prepare_registration_scenario(bundle, snapshots[0], player_name)
            analysis = analyze_tournament(
                bundle, scenario, player_name, iterations=iterations_box.value, progress=_progress
            )
            LAST_ANALYSES = [analysis]
            result_parts.append(_analysis_html(analysis))
        else:
            analyses = []
            skipped = []
            for position, snapshot in enumerate(snapshots, start=1):
                try:
                    status_html.value = f'<b>Мега {position}/{len(snapshots)}: {html.escape(snapshot.title)}</b>'
                    scenario = prepare_registration_scenario(bundle, snapshot, player_name)
                    analyses.append(analyze_tournament(
                        bundle, scenario, player_name, iterations=iterations_box.value, progress=_progress
                    ))
                except Exception as exc:
                    skipped.append(f'{snapshot.tour_id}: {exc}')
            LAST_ANALYSES = analyses
            summary = mega_summary(analyses)
            if summary.empty:
                raise RuntimeError('Не найдено ни одного подходящего турнира для игрока. ' + ' | '.join(skipped))
            summary_table = summary.rename(columns={
                'tour_id': '№', 'tournament': 'Турнир', 'start_date': 'Старт', 'category': 'Категория',
                'players': 'Игроков', 'win_probability_pct': 'Победа, %', 'expected_points': 'Матожидание очков',
                'entry_scenario': 'Сценарий',
                'official_grid': 'Официальная сетка',
            })[['№', 'Турнир', 'Старт', 'Категория', 'Игроков', 'Сценарий', 'Победа, %', 'Матожидание очков', 'Официальная сетка']].round(2)
            result_parts.extend(['<h3>Мега-режим: турниры по матожиданию очков</h3>', _table_html(summary_table)])
            if skipped:
                result_parts.append('<p><b>Пропущены:</b> ' + html.escape(' | '.join(skipped)) + '</p>')
            best_tour_id = str(summary.iloc[0]['tour_id'])
            best_analysis = next(item for item in analyses if str(item.snapshot.tour_id) == best_tour_id)
            result_parts.append(_analysis_html(best_analysis))
        result_html.value = ''.join(result_parts)
        status_html.value = '<b style="color:#177245">Расчёт завершён.</b>'
    except Exception as exc:
        message = html.escape(str(exc))
        status_html.value = f'<b style="color:#b00020">Ошибка: {message}</b>'
        result_html.value = f'<p style="color:#b00020"><b>Ошибка:</b> {message}</p>'
    finally:
        _RUNNING = False
        run_button.disabled = False

run_button._click_handlers.callbacks.clear()
run_button.on_click(_run)
_PREVIOUS_RUN_BUTTON = run_button
_PREVIOUS_RUN_HANDLER = _run
display(widgets.VBox([
    widgets.HTML('<h3>Параметры расчёта</h3>'), mode_box, tour_ids_box, player_box,
    widgets.HBox([iterations_box, refresh_box]), auto_mega_box, run_button, progress, status_html, result_html,
]))

In [6]:
try:
    _PREVIOUS_H2H_BUTTON._click_handlers.callbacks.clear()
except (NameError, AttributeError):
    pass

def _show_h2h(_):
    if LAST_ANALYSIS is None or not opponent_box.value:
        h2h_result_html.value = '<p>Сначала выполните расчёт турнира и выберите соперника.</p>'
        return
    opponent_id = str(opponent_box.value)
    opponent_name = LAST_ANALYSIS.simulation.player_names[opponent_id]
    target = LAST_ANALYSIS.target_player
    probability = LAST_ANALYSIS.probability_matrix.get((target.player_id, opponent_id))
    if probability is None:
        probability = 1.0 - LAST_ANALYSIS.probability_matrix[(opponent_id, target.player_id)]
    parts = [
        f'<h3>{html.escape(target.name)} — {html.escape(opponent_name)}</h3>',
        f'<p><b>{html.escape(target.name)}:</b> {100*probability:.2f}% &nbsp; ',
        f'<b>{html.escape(opponent_name)}:</b> {100*(1-probability):.2f}%</p>',
        '<p><a href="00_data_control_panel.ipynb" target="_blank">Открыть стандартную контрольную панель H2H</a></p>',
    ]
    detailed = pr.predict_match_by_names(
        bundle, target.name, opponent_name, LAST_ANALYSIS.prediction_date,
        context={
            'tournament_age_category': LAST_ANALYSIS.snapshot.age_group,
            'draw_type': 'Олимпийская',
            'tournament_name': LAST_ANALYSIS.snapshot.title,
            'tournament_city': '__UNKNOWN_CITY__',
        },
    )
    if detailed.get('ok'):
        parts.extend(['<h4>Карточки игроков из стандартного H2H</h4>', _table_html(pr.profiles_table(detailed))])
        factors = detailed.get('factor_contributions', pd.DataFrame())
        if not factors.empty:
            parts.extend(['<h4>Главные факторы</h4>', _table_html(factors.head(12))])
    else:
        parts.append('<p>Для одного из игроков пока нет полной карточки в историческом bundle; показана вероятность из турнирной матрицы.</p>')
    h2h_result_html.value = ''.join(parts)

h2h_button._click_handlers.callbacks.clear()
h2h_button.on_click(_show_h2h)
_PREVIOUS_H2H_BUTTON = h2h_button
_PREVIOUS_H2H_HANDLER = _show_h2h
display(widgets.VBox([widgets.HTML('<h3>Переход от сетки к Head-to-Head</h3>'), widgets.HBox([opponent_box, h2h_button]), h2h_result_html]))

## Источники правил

- Регламент РТТ на 2026 год: таблица №6 и пункты 13.3.1–13.3.3 (посев, секции и `X`).
- Официальные таблицы 4–9 классификационных очков ФТР/РТТ.

Ссылки на действующие документы также записаны в `rtt_predictor/tournament_simulation.py`.